# Chapter 10 — Mixture of Experts and Routing: Exercises

Eight exercises covering router computation, auxiliary loss, capacity overflow,
parameter counting, expert utilisation, all-to-all latency, MoE scaling, and Z-loss.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def fmt_vec(v, decimals=4):
    return "[" + ", ".join(f"{x:.{decimals}f}" for x in v) + "]"

def fmt_sci(x):
    if abs(x) >= 1e9: return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6: return f"{x/1e6:.2f}M"
    if abs(x) >= 1e3: return f"{x/1e3:.1f}K"
    return f"{x:.2f}"

def fmt_bytes(b):
    if b >= 1e12: return f"{b/1e12:.2f} TB"
    if b >= 1e9: return f"{b/1e9:.2f} GB"
    if b >= 1e6: return f"{b/1e6:.2f} MB"
    if b >= 1e3: return f"{b/1e3:.1f} KB"
    return f"{b:.0f} B"

def print_table(headers, rows, col_width=16):
    hdr = " | ".join(h.center(col_width) for h in headers)
    print(hdr)
    print("-" * len(hdr))
    for row in rows:
        print(" | ".join(str(c).center(col_width) for c in row))

print("Setup complete ✓")

---
## Exercise 1: Routing by Hand

Given router weights $W_g \in \mathbb{R}^{4 \times 3}$ and input $x \in \mathbb{R}^3$:

1. Compute router logits $z = W_g x$
2. Apply softmax to get gate probabilities $G(x)$
3. Select top-2 experts
4. Renormalise selected gates
5. Compute weighted expert output given expert outputs

**Given:**
$$W_g = \begin{bmatrix} 0.5 & -0.2 & 0.3 \\ -0.1 & 0.8 & 0.1 \\ 0.3 & 0.3 & -0.5 \\ -0.4 & 0.1 & 0.6 \end{bmatrix}, \quad x = \begin{bmatrix} 1.0 \\ 0.5 \\ -0.3 \end{bmatrix}$$

Expert outputs: $E_0(x) = [1.2, -0.5]$, $E_1(x) = [0.3, 0.8]$, $E_2(x) = [-0.1, 0.6]$, $E_3(x) = [0.7, 0.2]$

In [ ]:
# === Exercise 1: Scaffold ===
np.random.seed(42)

W_g = np.array([
    [0.5, -0.2, 0.3],
    [-0.1, 0.8, 0.1],
    [0.3, 0.3, -0.5],
    [-0.4, 0.1, 0.6]
])
x = np.array([1.0, 0.5, -0.3])

expert_outputs = {
    0: np.array([1.2, -0.5]),
    1: np.array([0.3, 0.8]),
    2: np.array([-0.1, 0.6]),
    3: np.array([0.7, 0.2]),
}

k = 2  # top-k

# Step 1: Compute logits z = W_g @ x
logits = None  # TODO

# Step 2: Softmax
probs = None  # TODO

# Step 3: Top-2 selection (indices of top-2 highest probabilities)
top_k_indices = None  # TODO

# Step 4: Renormalise
renorm_gates = None  # TODO

# Step 5: Weighted combination
moe_output = None  # TODO

print("Logits:", logits)
print("Softmax probs:", probs)
print(f"Top-{k} experts:", top_k_indices)
print("Renormalised gates:", renorm_gates)
print("MoE output:", moe_output)

In [ ]:
# === Exercise 1: Solution ===
np.random.seed(42)

W_g = np.array([
    [0.5, -0.2, 0.3],
    [-0.1, 0.8, 0.1],
    [0.3, 0.3, -0.5],
    [-0.4, 0.1, 0.6]
])
x = np.array([1.0, 0.5, -0.3])

expert_outputs = {
    0: np.array([1.2, -0.5]),
    1: np.array([0.3, 0.8]),
    2: np.array([-0.1, 0.6]),
    3: np.array([0.7, 0.2]),
}

k = 2

# Step 1: Compute logits
logits = W_g @ x
print("Step 1 — Logits z = W_g @ x:")
for i in range(4):
    terms = " + ".join(f"({W_g[i,j]:.1f})({x[j]:.1f})" for j in range(3))
    print(f"  z_{i} = {terms} = {logits[i]:.4f}")

# Step 2: Softmax
probs = softmax(logits)
print(f"\nStep 2 — Softmax: {fmt_vec(probs)}")
print(f"  Sum = {probs.sum():.6f}")

# Step 3: Top-2
top_k_indices = np.argsort(probs)[-k:][::-1]
top_k_probs = probs[top_k_indices]
print(f"\nStep 3 — Top-{k} experts: {top_k_indices}")
print(f"  Probabilities: {fmt_vec(top_k_probs)}")

# Step 4: Renormalise
renorm_gates = top_k_probs / top_k_probs.sum()
print(f"\nStep 4 — Renormalised gates: {fmt_vec(renorm_gates)}")
print(f"  Sum = {renorm_gates.sum():.6f}")

# Step 5: Weighted combination
moe_output = np.zeros(2)
for i, idx in enumerate(top_k_indices):
    moe_output += renorm_gates[i] * expert_outputs[idx]
    print(f"\nStep 5 — Expert {idx}: gate={renorm_gates[i]:.4f} × output={expert_outputs[idx]}")
    print(f"  Contribution: {renorm_gates[i] * expert_outputs[idx]}")

print(f"\nFinal MoE output: {fmt_vec(moe_output)}")

---
## Exercise 2: Auxiliary Loss Computation

Given a batch of $T=8$ tokens routed across 4 experts with the following statistics:
- $f = [0.50, 0.30, 0.15, 0.05]$ (fraction of tokens dispatched to each expert)
- $P = [0.40, 0.30, 0.20, 0.10]$ (mean gate probability per expert)

1. Compute $\mathcal{L}_{\text{aux}} = \alpha \cdot N \cdot \sum_i f_i \cdot P_i$ with $\alpha = 0.01$
2. Compute the loss for the perfectly **uniform** case
3. How much extra penalty does the imbalanced routing incur?

In [ ]:
# === Exercise 2: Scaffold ===
np.random.seed(42)

f = np.array([0.50, 0.30, 0.15, 0.05])  # load fractions
P = np.array([0.40, 0.30, 0.20, 0.10])  # mean probabilities
N = 4
k = 2
alpha = 0.01

# 1. Compute L_aux (actual)
L_aux = None  # TODO: alpha * N * sum(f_i * P_i)

# 2. Compute L_aux for uniform case
f_uniform = None  # TODO: what would f be if perfectly uniform?
P_uniform = None  # TODO: what would P be if perfectly uniform?
L_aux_uniform = None  # TODO

# 3. Penalty ratio
penalty_ratio = None  # TODO: L_aux / L_aux_uniform

print(f"L_aux (actual):  {L_aux}")
print(f"L_aux (uniform): {L_aux_uniform}")
print(f"Penalty ratio:   {penalty_ratio}")

In [ ]:
# === Exercise 2: Solution ===
np.random.seed(42)

f = np.array([0.50, 0.30, 0.15, 0.05])  # load fractions
P = np.array([0.40, 0.30, 0.20, 0.10])  # mean probabilities
N = 4
k = 2
alpha = 0.01

# 1. Compute L_aux (actual)
print("Step 1: Compute L_aux = α · N · Σ(f_i · P_i)")
products = f * P
for i in range(N):
    print(f"  f_{i} · P_{i} = {f[i]:.2f} × {P[i]:.2f} = {products[i]:.4f}")
product_sum = products.sum()
print(f"  Σ(f_i · P_i) = {product_sum:.4f}")

L_aux = alpha * N * product_sum
print(f"  L_aux = {alpha} × {N} × {product_sum:.4f} = {L_aux:.6f}")

# 2. Uniform case
print(f"\nStep 2: Uniform case")
# If k=2 and N=4, each expert gets k/N = 0.5 fraction of dispatches
f_uniform = np.ones(N) * (k / N)  # = [0.5, 0.5, 0.5, 0.5]
P_uniform = np.ones(N) / N        # = [0.25, 0.25, 0.25, 0.25]
print(f"  f_uniform = {fmt_vec(f_uniform)}")
print(f"  P_uniform = {fmt_vec(P_uniform)}")

L_aux_uniform = alpha * N * np.sum(f_uniform * P_uniform)
print(f"  Σ(f_i · P_i) = {np.sum(f_uniform * P_uniform):.4f}")
print(f"  L_aux_uniform = {L_aux_uniform:.6f}")

# 3. Penalty ratio
penalty_ratio = L_aux / L_aux_uniform
print(f"\nStep 3: Penalty ratio = {L_aux:.6f} / {L_aux_uniform:.6f} = {penalty_ratio:.4f}")
print(f"  Imbalanced routing incurs {(penalty_ratio - 1)*100:.1f}% extra penalty")
print(f"  Cauchy-Schwarz: Σ(f·P) ≥ (Σf)(ΣP)/N when f,P have same ranking")
print(f"  → Aux loss is minimised when f and P are uncorrelated or uniform")

---
## Exercise 3: Capacity and Overflow

Configuration: $T=1024$ tokens, $N=8$ experts, $k=2$, $CF=1.25$.

1. Compute the expert capacity $C = \lceil CF \cdot T \cdot k / N \rceil$
2. If expert 1 receives 200 tokens and expert 3 receives 400 tokens, how many are dropped per expert?
3. What is the minimum CF needed to handle this load without dropping from expert 3?

In [ ]:
# === Exercise 3: Scaffold ===
np.random.seed(42)

T = 1024
N = 8
k = 2
CF = 1.25

# 1. Compute capacity
C = None  # TODO: ceil(CF * T * k / N)

# 2. Tokens dropped
expert_1_tokens = 200
expert_3_tokens = 400
dropped_expert_1 = None  # TODO: max(0, tokens - C)
dropped_expert_3 = None  # TODO

# 3. Minimum CF for expert 3
min_CF = None  # TODO: expert_3_tokens * N / (T * k)

print(f"Capacity C = {C}")
print(f"Expert 1: {expert_1_tokens} tokens → {dropped_expert_1} dropped")
print(f"Expert 3: {expert_3_tokens} tokens → {dropped_expert_3} dropped")
print(f"Min CF for no overflow at expert 3: {min_CF}")

In [ ]:
# === Exercise 3: Solution ===
np.random.seed(42)

T = 1024
N = 8
k = 2
CF = 1.25

# 1. Compute capacity
ideal_tokens_per_expert = T * k / N
print(f"Step 1: Ideal tokens per expert = T × k / N = {T} × {k} / {N} = {ideal_tokens_per_expert:.0f}")

C = int(np.ceil(CF * T * k / N))
print(f"  C = ceil(CF × T × k / N) = ceil({CF} × {T} × {k} / {N}) = ceil({CF * T * k / N}) = {C}")
print(f"  Total capacity across all experts: {C * N} slots for {T * k} dispatches")
print(f"  Overhead: {(C * N - T * k) / (T * k) * 100:.1f}%")

# 2. Token dropping
print(f"\nStep 2: Token dropping with C = {C}")

expert_1_tokens = 200
expert_3_tokens = 400

dropped_expert_1 = max(0, expert_1_tokens - C)
dropped_expert_3 = max(0, expert_3_tokens - C)

print(f"  Expert 1: {expert_1_tokens} tokens, capacity {C}")
print(f"    → Dropped: max(0, {expert_1_tokens} - {C}) = {dropped_expert_1}")
print(f"    → Processed: {min(expert_1_tokens, C)}")

print(f"  Expert 3: {expert_3_tokens} tokens, capacity {C}")
print(f"    → Dropped: max(0, {expert_3_tokens} - {C}) = {dropped_expert_3}")
print(f"    → Processed: {min(expert_3_tokens, C)}")
print(f"    → Drop rate: {dropped_expert_3/expert_3_tokens*100:.1f}%")

# 3. Minimum CF
print(f"\nStep 3: Minimum CF for expert 3")
min_CF = expert_3_tokens * N / (T * k)
print(f"  Need C ≥ {expert_3_tokens}")
print(f"  CF ≥ C × N / (T × k) = {expert_3_tokens} × {N} / ({T} × {k}) = {min_CF:.4f}")
print(f"  Expert 3 has {expert_3_tokens/ideal_tokens_per_expert:.1f}× of ideal load")
print(f"  This extreme imbalance (>CF=1.25) indicates routing problem")

---
## Exercise 4: Parameter Count — Dense vs MoE

Dense model: $d_{\text{model}}=4096$, $d_{ff}=16384$ (standard FFN per layer).
MoE replacement: $N=8$ experts, each with same $d_{ff}=16384$.

1. Compute total FFN params for dense vs MoE (one layer, ignore biases)
2. Compute active params per token for $k=2$
3. What is the effective "parameter amplification" factor?

In [ ]:
# === Exercise 4: Scaffold ===
np.random.seed(42)

d_model = 4096
d_ff = 16384
N = 8
k = 2

# 1. Total FFN params
dense_params = None    # TODO: 2 * d_model * d_ff (W1 and W2)
moe_params = None      # TODO: N * dense_params + router
router_params = None   # TODO: N * d_model

# 2. Active params per token
active_params = None   # TODO: k * (params per expert) + router

# 3. Amplification factor
amplification = None   # TODO: total MoE / active

print(f"Dense FFN params:      {dense_params}")
print(f"MoE total FFN params:  {moe_params}")
print(f"MoE active params:     {active_params}")
print(f"Amplification factor:  {amplification}")

In [ ]:
# === Exercise 4: Solution ===
np.random.seed(42)

d_model = 4096
d_ff = 16384
N = 8
k = 2

# 1. Total FFN params per layer
print("Step 1: Parameter count (one layer, ignoring biases)")
print(f"  Dense FFN: W1 ∈ R^{{d_ff × d_model}} + W2 ∈ R^{{d_model × d_ff}}")

params_per_expert = 2 * d_model * d_ff  # W1 + W2
dense_params = params_per_expert
router_params = N * d_model
moe_params = N * params_per_expert + router_params

print(f"  Params per expert: 2 × {d_model} × {d_ff} = {fmt_sci(params_per_expert)}")
print(f"  Dense total: {fmt_sci(dense_params)}")
print(f"  MoE total: {N} × {fmt_sci(params_per_expert)} + {fmt_sci(router_params)} (router) = {fmt_sci(moe_params)}")
print(f"  Ratio: {moe_params/dense_params:.1f}× more params in MoE")

# 2. Active params per token
print(f"\nStep 2: Active params per token (k={k})")
active_params = k * params_per_expert + router_params
print(f"  Active: {k} × {fmt_sci(params_per_expert)} + {fmt_sci(router_params)} = {fmt_sci(active_params)}")
print(f"  Active fraction: {active_params/moe_params*100:.1f}%")

# 3. Amplification
print(f"\nStep 3: Parameter amplification")
amplification = moe_params / active_params
print(f"  Amplification = total / active = {fmt_sci(moe_params)} / {fmt_sci(active_params)} = {amplification:.2f}×")
print(f"  Ideal amplification (router negligible): N/k = {N}/{k} = {N/k:.1f}×")

# Comparison with real model
print(f"\nReal-world comparison (Mixtral 8×7B):")
print(f"  Total params: ~46.7B")
print(f"  Active params: ~12.9B")
print(f"  Amplification: ~3.6× (includes attention, embeddings)")

# Memory vs compute
print(f"\nMemory vs compute per layer (BF16):")
print(f"  Dense memory:  {fmt_bytes(dense_params * 2)}")
print(f"  MoE memory:    {fmt_bytes(moe_params * 2)}")
print(f"  MoE compute:   same as dense with k={k} experts at {fmt_sci(params_per_expert)} each")

---
## Exercise 5: Expert Utilisation and Gini Coefficient

Given expert load distribution for 8 experts:
$$f = [0.35, 0.25, 0.15, 0.10, 0.08, 0.04, 0.02, 0.01]$$

1. Compute the Gini coefficient: $G = \frac{\sum_i \sum_j |f_i - f_j|}{2N \sum_i f_i}$
2. Is this a healthy load distribution? (G < 0.15 = healthy, 0.15–0.35 = warning, > 0.35 = critical)
3. How many experts are receiving less than half the ideal load ($1/N$)?

In [ ]:
# === Exercise 5: Scaffold ===
np.random.seed(42)

f = np.array([0.35, 0.25, 0.15, 0.10, 0.08, 0.04, 0.02, 0.01])
N = 8

# 1. Gini coefficient
# G = sum_i sum_j |f_i - f_j| / (2 * N * sum(f))
gini = None  # TODO

# 2. Health assessment
health = None  # TODO: based on Gini thresholds

# 3. Underloaded experts
ideal_load = 1.0 / N
underloaded = None  # TODO: count experts with f_i < ideal_load / 2

print(f"Gini coefficient: {gini}")
print(f"Health: {health}")
print(f"Underloaded experts: {underloaded}")

In [ ]:
# === Exercise 5: Solution ===
np.random.seed(42)

f = np.array([0.35, 0.25, 0.15, 0.10, 0.08, 0.04, 0.02, 0.01])
N = 8

# 1. Gini coefficient
print("Step 1: Gini coefficient computation")
print(f"  Load distribution f = {fmt_vec(f)}")
print(f"  Sum of f = {f.sum():.2f}")

diff_sum = 0
for i in range(N):
    for j in range(N):
        diff_sum += abs(f[i] - f[j])

gini = diff_sum / (2 * N * f.sum())
print(f"  Σ_i Σ_j |f_i - f_j| = {diff_sum:.4f}")
print(f"  G = {diff_sum:.4f} / (2 × {N} × {f.sum():.2f}) = {gini:.4f}")

# 2. Health assessment
print(f"\nStep 2: Health assessment")
if gini < 0.15:
    health = "✓ Healthy"
elif gini < 0.35:
    health = "⚠ Warning"
else:
    health = "✗ Critical"
print(f"  Gini = {gini:.4f} → {health}")
print(f"  Thresholds: healthy < 0.15, warning 0.15–0.35, critical > 0.35")

# 3. Underloaded experts
print(f"\nStep 3: Underloaded experts")
ideal_load = 1.0 / N
threshold = ideal_load / 2
print(f"  Ideal load per expert: 1/{N} = {ideal_load:.4f}")
print(f"  Underload threshold (50% of ideal): {threshold:.4f}")

underloaded = 0
for i in range(N):
    status = "UNDERLOADED" if f[i] < threshold else ("OVERLOADED" if f[i] > 2 * ideal_load else "OK")
    print(f"  Expert {i}: f={f[i]:.3f} → {status}")
    if f[i] < threshold:
        underloaded += 1

print(f"\n  {underloaded} experts below 50% of ideal load")
print(f"  Recommendation: increase α, add expert dropout, or reinitialise dead experts")

---
## Exercise 6: All-to-All Latency

Setup: $EP=32$ GPUs, $T=4096$ tokens, $d=4096$, BF16 (2 bytes), NVLink 600 GB/s.

1. How much data does each GPU send in one all-to-all (assuming uniform load)?
2. Estimate one-way all-to-all time
3. Total all-to-all time per MoE layer (dispatch + combine) for 32 MoE layers
4. Compare to expert compute time (assume 312 TFLOPS H100, 8 experts per GPU, d_ff=14336)

In [ ]:
# === Exercise 6: Scaffold ===
np.random.seed(42)

EP = 32
T = 4096
d = 4096
bytes_per_elem = 2  # BF16
bw = 600e9          # 600 GB/s NVLink

# 1. Data per GPU
tokens_per_gpu = T / EP
data_per_gpu = None  # TODO: tokens_per_gpu * d * bytes_per_elem

# 2. One-way time
time_one_way_ms = None  # TODO: data_per_gpu / bw * 1000

# 3. Total time per model
n_moe_layers = 32
total_comm_ms = None  # TODO: 2 * time_one_way * n_moe_layers

# 4. Expert compute comparison
d_ff = 14336
k = 2
experts_per_gpu = 8  # N_total=256, EP=32
tflops = 312  # H100 BF16
compute_time_ms = None  # TODO

print(f"Data per GPU: {data_per_gpu}")
print(f"One-way time: {time_one_way_ms} ms")
print(f"Total comm time: {total_comm_ms} ms")
print(f"Expert compute time: {compute_time_ms} ms")

In [ ]:
# === Exercise 6: Solution ===
np.random.seed(42)

EP = 32
T = 4096
d = 4096
bytes_per_elem = 2  # BF16
bw = 600e9          # 600 GB/s NVLink

# 1. Data per GPU
print("Step 1: Data per GPU in one all-to-all")
tokens_per_gpu = T / EP
print(f"  Tokens per GPU: {T}/{EP} = {tokens_per_gpu:.0f}")

# Each GPU sends its tokens to the GPUs holding the target experts
# With uniform routing: each GPU sends T/EP tokens × d × bytes to other GPUs
data_per_gpu = tokens_per_gpu * d * bytes_per_elem
print(f"  Data per GPU: {tokens_per_gpu:.0f} × {d} × {bytes_per_elem} = {fmt_bytes(data_per_gpu)}")

# 2. One-way time
print(f"\nStep 2: One-way all-to-all time")
time_one_way_s = data_per_gpu / bw
time_one_way_ms = time_one_way_s * 1000
print(f"  Time = {fmt_bytes(data_per_gpu)} / {bw/1e9:.0f} GB/s = {time_one_way_ms:.4f} ms")

# 3. Total communication time
print(f"\nStep 3: Total communication time")
n_moe_layers = 32
time_per_layer = 2 * time_one_way_ms  # dispatch + combine
total_comm_ms = time_per_layer * n_moe_layers
print(f"  Per layer: 2 × {time_one_way_ms:.4f} = {time_per_layer:.4f} ms")
print(f"  {n_moe_layers} layers: {total_comm_ms:.3f} ms")

# 4. Expert compute comparison
print(f"\nStep 4: Expert compute comparison")
d_ff = 14336
k = 2
N_total = 256
experts_per_gpu = N_total // EP  # = 8
tokens_per_expert = T * k / N_total  # uniform
tflops = 312  # H100 BF16

# FLOPs per expert per forward: 2 * d * d_ff * tokens_per_expert
flops_per_expert = 2 * d * d_ff * tokens_per_expert
flops_per_gpu = flops_per_expert * experts_per_gpu
compute_time_ms = flops_per_gpu / (tflops * 1e12) * 1000

print(f"  Experts per GPU: {experts_per_gpu}")
print(f"  Tokens per expert: {tokens_per_expert:.0f}")
print(f"  FLOPs per GPU: {flops_per_gpu/1e9:.2f} GFLOPs")
print(f"  Compute time: {compute_time_ms:.4f} ms")

comm_compute_ratio = time_per_layer / compute_time_ms
print(f"\n  Communication/Compute ratio: {comm_compute_ratio:.2f}")
if comm_compute_ratio < 1:
    print("  → Communication fully overlappable with compute ✓")
else:
    print("  → Communication bottleneck — need overlap strategies")

---
## Exercise 7: MoE vs Dense Scaling

A dense model has 10B active parameters. An MoE variant uses the same active
compute but with $N=64$ experts ($k=2$), giving $10B \times 64/2 = 320B$ total params.

1. Using simplified scaling $\text{quality} \propto \log(N_{\text{total}} / k)$,
   estimate relative quality improvement
2. Is the 32× memory overhead worth the quality gain?
3. What is the memory requirement in BF16 for both models?

In [ ]:
# === Exercise 7: Scaffold ===
np.random.seed(42)

N_active = 10e9
N_experts = 64
k = 2

# 1. Quality scaling
N_total_moe = None  # TODO: N_active * N_experts / k
quality_dense = None   # TODO: log(N_active)
quality_moe = None     # TODO: log(N_total_moe)
improvement = None     # TODO: relative improvement

# 2. Memory overhead
memory_dense = None  # TODO: N_active * 2 bytes (BF16)
memory_moe = None    # TODO: N_total * 2 bytes
memory_ratio = None  # TODO

print(f"Total MoE params: {N_total_moe}")
print(f"Quality improvement: {improvement}")
print(f"Memory ratio: {memory_ratio}")

In [ ]:
# === Exercise 7: Solution ===
np.random.seed(42)

N_active = 10e9
N_experts = 64
k = 2

# 1. Quality scaling
print("Step 1: Quality scaling estimate")
N_total_moe = N_active * N_experts / k
print(f"  MoE total params: {fmt_sci(N_active)} × {N_experts}/{k} = {fmt_sci(N_total_moe)}")

# Simplified: quality bonus ∝ log(N_total / N_active)
# Dense: log(1) = 0 bonus; MoE: log(N/k) bonus
log_scaling_bonus = np.log(N_experts / k)
print(f"  Log scaling bonus: log({N_experts}/{k}) = log({N_experts//k}) = {log_scaling_bonus:.3f}")

# Compare to different expert counts
print(f"\n  Scaling comparison at {fmt_sci(N_active)} active:")
for n_exp in [1, 4, 8, 16, 64, 128, 256]:
    total = N_active * n_exp / k
    bonus = np.log(n_exp / k)
    print(f"    N={n_exp:3d}, total={fmt_sci(total):>8s}, bonus=log({n_exp//k:>3d})={bonus:.3f}")

# 2. Memory overhead
print(f"\nStep 2: Memory analysis (BF16 = 2 bytes/param)")
memory_dense = N_active * 2
memory_moe = N_total_moe * 2
memory_ratio = memory_moe / memory_dense

print(f"  Dense model: {fmt_sci(N_active)} params → {fmt_bytes(memory_dense)}")
print(f"  MoE model:   {fmt_sci(N_total_moe)} params → {fmt_bytes(memory_moe)}")
print(f"  Memory ratio: {memory_ratio:.1f}×")

gpus_80gb = int(np.ceil(memory_moe / (80e9)))
gpus_dense = int(np.ceil(memory_dense / (80e9)))
print(f"\n  GPUs needed (80 GB H100):")
print(f"    Dense: {gpus_dense} GPU(s)")
print(f"    MoE:   {gpus_80gb} GPU(s)")

print(f"\nStep 3: Worth the overhead?")
print(f"  Quality gain: log-scale bonus of {log_scaling_bonus:.3f}")
print(f"  Memory cost:  {memory_ratio:.0f}× more GPU memory")
print(f"  Compute cost: same (only k={k} experts active)")
print(f"  Verdict: MoE provides quality of ~{fmt_sci(N_total_moe)} dense model")
print(f"           at compute cost of {fmt_sci(N_active)} dense model")
print(f"           → strong value if memory/multi-GPU infrastructure available")

---
## Exercise 8: Z-Loss Calculation

Router logits for 4 experts: $z = [3.2, 1.1, -0.5, 2.8]$.

1. Compute the Z-loss: $\mathcal{L}_z = \left(\log \sum_i e^{z_i}\right)^2$
2. Compute Z-loss for uniform logits $z_{\text{uniform}} = [1, 1, 1, 1]$
3. Why does Z-loss penalise the first case more? What is the regularisation effect?

In [ ]:
# === Exercise 8: Scaffold ===
np.random.seed(42)

z = np.array([3.2, 1.1, -0.5, 2.8])
z_uniform = np.array([1.0, 1.0, 1.0, 1.0])

# 1. Z-loss for z
L_z = None  # TODO: (log(sum(exp(z_i))))^2

# 2. Z-loss for z_uniform
L_z_uniform = None  # TODO

# 3. Interpretation
# TODO: compute softmax of both; compare

print(f"Z-loss (z):       {L_z}")
print(f"Z-loss (uniform): {L_z_uniform}")

In [ ]:
# === Exercise 8: Solution ===
np.random.seed(42)

z = np.array([3.2, 1.1, -0.5, 2.8])
z_uniform = np.array([1.0, 1.0, 1.0, 1.0])

# 1. Z-loss for z
print("Step 1: Z-loss for z = [3.2, 1.1, -0.5, 2.8]")
exp_z = np.exp(z)
for i, (zi, ei) in enumerate(zip(z, exp_z)):
    print(f"  exp({zi:.1f}) = {ei:.4f}")
sum_exp = exp_z.sum()
log_sum_exp = np.log(sum_exp)
L_z = log_sum_exp ** 2
print(f"  Σ exp(z_i) = {sum_exp:.4f}")
print(f"  log(Σ exp(z_i)) = {log_sum_exp:.4f}")
print(f"  L_z = ({log_sum_exp:.4f})² = {L_z:.4f}")

# 2. Z-loss for uniform
print(f"\nStep 2: Z-loss for z_uniform = [1, 1, 1, 1]")
exp_u = np.exp(z_uniform)
sum_exp_u = exp_u.sum()
log_sum_exp_u = np.log(sum_exp_u)
L_z_uniform = log_sum_exp_u ** 2
print(f"  Σ exp(z_i) = 4 × exp(1) = {sum_exp_u:.4f}")
print(f"  log(Σ exp(z_i)) = {log_sum_exp_u:.4f}")
print(f"  L_z = ({log_sum_exp_u:.4f})² = {L_z_uniform:.4f}")

# 3. Interpretation
print(f"\nStep 3: Interpretation")
print(f"  Z-loss ratio: {L_z/L_z_uniform:.2f}× higher for peaked logits")

probs_z = softmax(z)
probs_u = softmax(z_uniform)
print(f"\n  Softmax of z:         {fmt_vec(probs_z)}")
print(f"  Softmax of z_uniform: {fmt_vec(probs_u)}")

entropy_z = -np.sum(probs_z * np.log2(probs_z + 1e-15))
entropy_u = -np.sum(probs_u * np.log2(probs_u + 1e-15))
print(f"\n  Routing entropy (z):       {entropy_z:.4f} bits")
print(f"  Routing entropy (uniform): {entropy_u:.4f} bits (maximum)")

print(f"\n  Why Z-loss penalises more:")
print(f"  - Large logit magnitudes → near-one-hot softmax → hard routing")
print(f"  - Hard routing → less exploration → expert collapse risk")
print(f"  - Z-loss ∝ (logsumexp)² grows with logit scale")
print(f"  - Encourages moderate logit values → softer routing → stability")
print(f"  - Key: Z-loss penalises the MAGNITUDE, not the distribution shape")
print(f"  - Logits [0.32, 0.11, -0.05, 0.28] give SAME softmax but lower Z-loss")

# Verify
z_scaled = z * 0.1
L_z_scaled = np.log(np.exp(z_scaled).sum()) ** 2
print(f"\n  Scaled z (×0.1): Z-loss = {L_z_scaled:.4f} (vs {L_z:.4f} for original)")
print(f"  Same relative routing, but {L_z/L_z_scaled:.1f}× lower Z-loss")

print("\n" + "="*60)
print("ALL EXERCISES COMPLETE")
print("="*60)